# CXformer-Small: Kaggle Tesla T4 GPU Pipeline Verification & Memory Benchmark

This controlled notebook verifies the **CXformer-Small (Dinov2WithRegistersModel)** transfer learning pipeline on a Kaggle NVIDIA Tesla T4 GPU.

### Objectives:
1. Verify CUDA availability and Tesla T4 GPU device identity.
2. Verify mounted paths for the prepared chest X-ray dataset and CXformer checkpoint.
3. Verify official CXformer `CustomBitImageProcessor` and OpenCV (`cv2`) loading.
4. Execute the pipeline smoke test (`scripts/train.py --smoke_test`).
5. Measure empirical peak GPU memory and step latency for batch sizes 2, 4, 8, and 16.
6. Execute a controlled 1-epoch training trial and verify checkpoint save/reload integrity.

> **Safe Execution Notice:** All tests run with the pretrained backbone strictly frozen (`requires_grad=False`). Only the classification head (`LayerNorm -> Dropout(0.3) -> Linear(384, 2)`) is trained.

In [ ]:
# Cell 1: Environment Diagnostics & GPU Identity Verification
import sys
import os
import torch

print('=' * 65)
print(' SYSTEM & GPU DIAGNOSTICS')
print('=' * 65)
print(f'Python version   : {sys.version.split()[0]}')
print(f'PyTorch version  : {torch.__version__}')
print(f'CUDA Available   : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    device = torch.device('cuda:0')
    gpu_name = torch.cuda.get_device_name(0)
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'Active Device    : {device}')
    print(f'GPU Identity     : {gpu_name}')
    print(f'Total VRAM       : {total_mem_gb:.2f} GB')
    assert total_mem_gb >= 14.0, f'Expected at least 14GB VRAM for Tesla T4, got {total_mem_gb:.2f} GB'
    print('[PASS] Tesla T4 GPU successfully detected and verified.')
else:
    raise SystemError('CUDA GPU not detected! Please enable GPU in Kaggle settings: Notebook Settings -> Accelerator -> GPU T4 x1.')


## 2. Unpack Project Source Code

Unpack the lightweight source bundle (`gpu_network_source.zip`) into `/kaggle/working/gpu-network`.

In [ ]:
# Cell 2: Setup Project Workspace and Python Path
import zipfile

WORK_DIR = '/kaggle/working/gpu-network'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

zip_candidates = [
    '/kaggle/working/gpu_network_source.zip',
    '/kaggle/input/gpu-network-source/gpu_network_source.zip',
    '/kaggle/input/gpu_network_source.zip',
]

unpacked = False
for zc in zip_candidates:
    if os.path.isfile(zc):
        with zipfile.ZipFile(zc, 'r') as zf:
            zf.extractall(WORK_DIR)
        print(f'[+] Unpacked source archive from: {zc}')
        unpacked = True
        break

if not unpacked:
    if os.path.isdir(os.path.join(WORK_DIR, 'src')):
        print(f'[+] Project source files already present in {WORK_DIR}')
    else:
        print('[!] Note: gpu_network_source.zip not found in default search paths.')
        print(f'Please ensure src, scripts, and configs folders exist in {WORK_DIR}')

if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

print('Working Directory:', os.getcwd())
print('Directory Contents:', os.listdir(WORK_DIR))


## 3. Verify Mounted Dataset and Pretrained Model Paths

Validates the directory structures for both the prepared chest X-ray dataset and the CXformer-Small checkpoint files.

In [ ]:
# Cell 3: Verify Mounted Dataset & CXformer Weights
# 1. Prepared Dataset Resolution
expected_data_paths = [
    '/kaggle/input/datasets/vishwad007/chest-xray-prepared/prepared',
    '/kaggle/input/chest-xray-prepared/prepared',
    '/kaggle/input/chest-xray-prepared',
    '/kaggle/working/prepared',
]

DATA_ROOT = None
for p in expected_data_paths:
    if os.path.isdir(p):
        splits = [s for s in ['train', 'val', 'test'] if os.path.isdir(os.path.join(p, s))]
        if len(splits) == 3:
            DATA_ROOT = p
            break

if DATA_ROOT is None:
    # Dynamic scan under /kaggle/input
    for root, dirs, files in os.walk('/kaggle/input'):
        splits = [s for s in ['train', 'val', 'test'] if os.path.isdir(os.path.join(root, s))]
        if len(splits) == 3:
            DATA_ROOT = root
            break

assert DATA_ROOT is not None, 'Failed to locate prepared dataset directory under /kaggle/input!'
print(f'[PASS] Resolved Dataset Root: {DATA_ROOT}')
for split in ['train', 'val', 'test']:
    n_count = len(os.listdir(os.path.join(DATA_ROOT, split, 'NORMAL')))
    p_count = len(os.listdir(os.path.join(DATA_ROOT, split, 'PNEUMONIA')))
    print(f'  - {split.upper():5s}: {n_count} NORMAL, {p_count} PNEUMONIA ({n_count + p_count} total)')

# 2. CXformer Model Files Resolution
expected_model_paths = [
    '/kaggle/input/datasets/vishwad007/cxformer-small-files',
    '/kaggle/input/cxformer-small-files',
    '/kaggle/input/cxformer-small',
    '/kaggle/working/CXformer_small_files',
]

MODEL_DIR = None
for p in expected_model_paths:
    if os.path.isdir(p) and os.path.isfile(os.path.join(p, 'model.safetensors')) and os.path.isfile(os.path.join(p, 'config.json')):
        MODEL_DIR = p
        break

if MODEL_DIR is None:
    # Dynamic scan under /kaggle/input
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'model.safetensors' in files and 'config.json' in files:
            MODEL_DIR = root
            break

assert MODEL_DIR is not None, 'Failed to locate valid CXformer-Small checkpoint under /kaggle/input!'
print(f'[PASS] Resolved CXformer Model Directory: {MODEL_DIR}')
print('  Found model files:', [f for f in os.listdir(MODEL_DIR) if not f.startswith('.')])


## 4. Verify Official CXformer Image Processor & Histogram Equalization

Verifies that `cv2` and `CustomBitImageProcessor` load properly from `MODEL_DIR` with `do_hist_equalization=True` and produces `(3, 518, 518)` tensors on a real image.

In [ ]:
# Cell 4: Verify Official Custom Image Processor
import cv2
print(f'OpenCV Version: {cv2.__version__}')

from src.data.dataset import get_cxformer_processor, CXformerTransform
from PIL import Image

processor = get_cxformer_processor(MODEL_DIR)
print(f'Loaded Processor Class  : {processor.__class__.__name__}')
print(f'Histogram Equalization  : {getattr(processor, "do_hist_equalization", False)}')
assert getattr(processor, 'do_hist_equalization', False) is True, 'Histogram equalization is NOT enabled!'

# Test preprocessing on a real training image
normal_dir = os.path.join(DATA_ROOT, 'train', 'NORMAL')
sample_img_path = os.path.join(normal_dir, os.listdir(normal_dir)[0])
transform = CXformerTransform(processor=processor)

with Image.open(sample_img_path) as img:
    sample_tensor = transform(img)

print(f'Sample Tensor Shape     : {tuple(sample_tensor.shape)}')
print(f'Sample Tensor Dtype     : {sample_tensor.dtype}')
print(f'Sample Tensor Min/Max   : {sample_tensor.min().item():.4f} / {sample_tensor.max().item():.4f}')
assert sample_tensor.shape == (3, 518, 518)
assert sample_tensor.dtype == torch.float32
assert torch.isfinite(sample_tensor).all().item() is True
print('[PASS] Official CXformer processor verified on real chest X-ray.')


## 5. Execute Existing Smoke Test on Tesla T4

Runs `scripts/train.py --model cxformer_small --smoke_test --batch_size 2` using the verified paths.

In [ ]:
# Cell 5: Run Existing Smoke Test via CLI
import subprocess

cmd = [
    sys.executable, 'scripts/train.py',
    '--model', 'cxformer_small',
    '--data_dir', DATA_ROOT,
    '--model_dir', MODEL_DIR,
    '--batch_size', '2',
    '--smoke_test'
]

print('Executing:', ' '.join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout)
if res.stderr:
    print('STDERR:\n', res.stderr)

assert res.returncode == 0, f'Smoke test failed with return code {res.returncode}'
print('[PASS] CXformer-Small smoke test passed successfully on Tesla T4 GPU.')


## 6. Systematic GPU Peak Memory & Latency Benchmark

Measures empirical peak VRAM (`torch.cuda.max_memory_allocated()`) and per-step execution latency across mini-batch sizes **2, 4, 8, and 16**.
- Performs real forward pass, weighted cross-entropy loss, backward pass, and AdamW optimizer step on the classification head.
- Verifies that the backbone remains strictly frozen with zero gradients.
- Catches and reports Out-Of-Memory (OOM) errors without crashing.

In [ ]:
# Cell 6: Systematic GPU Peak Memory & Latency Benchmark
import time
import gc
import torch
import torch.nn as nn
from src.models import build_cxformer_small
from src.data.dataset import create_cxformer_data_loaders, compute_class_weights

print('=' * 75)
print(' CXFORMER-SMALL TESLA T4 GPU MEMORY & LATENCY BENCHMARK')
print('=' * 75)

device = torch.device('cuda:0')
test_batch_sizes = [2, 4, 8, 16]
benchmark_results = []

for bs in test_batch_sizes:
    print(f'\n--- Testing Batch Size: {bs} ---')
    
    # Reset CUDA memory stats and clear cache
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device)
    
    try:
        # 1. Instantiate model and move to T4 GPU
        model = build_cxformer_small(model_dir=MODEL_DIR, num_classes=2, freeze_backbone=True).to(device)
        model.train()
        
        # 2. Build DataLoaders
        train_loader, _, _ = create_cxformer_data_loaders(
            prepared_root=DATA_ROOT,
            batch_size=bs,
            cxformer_model_dir=MODEL_DIR,
            num_workers=2,
        )
        
        # 3. Setup dynamic weighted loss & AdamW optimizer
        train_labels = train_loader.dataset.get_labels()
        weights = compute_class_weights(train_labels, num_classes=2, device=device)
        criterion = nn.CrossEntropyLoss(weight=weights)
        optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)
        
        # 4. Load real training batch
        images, targets = next(iter(train_loader))
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        
        # 5. Warm-up pass
        optimizer.zero_grad()
        out = model(images)
        loss = criterion(out, targets)
        loss.backward()
        optimizer.step()
        
        # 6. Timed benchmark step
        torch.cuda.synchronize(device)
        t_start = time.time()
        
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        
        torch.cuda.synchronize(device)
        t_elapsed = time.time() - t_start
        
        # 7. Record peak memory
        peak_bytes = torch.cuda.max_memory_allocated(device)
        peak_mb = peak_bytes / (1024**2)
        peak_gb = peak_bytes / (1024**3)
        throughput = bs / t_elapsed
        
        # 8. Verify gradient isolation
        for name, p in model.backbone.named_parameters():
            assert p.grad is None, f'Backbone parameter {name} received gradients!'
        for name, p in model.classifier.named_parameters():
            assert p.grad is not None, f'Classification head parameter {name} has no gradient!'
        
        print(f'  [SUCCESS] Batch Size {bs:2d} | Step Time: {t_elapsed:.3f}s | Throughput: {throughput:.1f} img/s | Peak VRAM: {peak_mb:.1f} MB ({peak_gb:.2f} GB)')
        
        benchmark_results.append({
            'batch_size': bs,
            'status': 'PASS',
            'step_time_sec': round(t_elapsed, 4),
            'throughput_img_sec': round(throughput, 2),
            'peak_vram_mb': round(peak_mb, 1),
            'peak_vram_gb': round(peak_gb, 3),
        })
    except torch.cuda.OutOfMemoryError:
        print(f'  [OOM ERROR] Mini-batch size {bs} exceeded available VRAM (16 GB)!')
        benchmark_results.append({
            'batch_size': bs,
            'status': 'OOM',
            'step_time_sec': None,
            'throughput_img_sec': None,
            'peak_vram_mb': None,
            'peak_vram_gb': None,
        })
        torch.cuda.empty_cache()
    except Exception as e:
        print(f'  [ERROR] Mini-batch size {bs} failed with exception: {e}')
        benchmark_results.append({
            'batch_size': bs,
            'status': f'FAILED: {str(e)}',
            'step_time_sec': None,
            'throughput_img_sec': None,
            'peak_vram_mb': None,
            'peak_vram_gb': None,
        })
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()

# Summary Table Output
print('\n' + '=' * 75)
print(f'{"Batch Size":<12} {"Status":<10} {"Step Time (s)":<15} {"Throughput (img/s)":<22} {"Peak VRAM":<15}')
print('-' * 75)
for r in benchmark_results:
    vram_str = f"{r['peak_vram_mb']:.1f} MB ({r['peak_vram_gb']:.2f} GB)" if r['peak_vram_mb'] else 'N/A'
    t_str = f"{r['step_time_sec']:.3f}" if r['step_time_sec'] else 'N/A'
    th_str = f"{r['throughput_img_sec']:.1f}" if r['throughput_img_sec'] else 'N/A'
    print(f"{r['batch_size']:<12} {r['status']:<10} {t_str:<15} {th_str:<22} {vram_str:<15}")
print('=' * 75)


## 7. Controlled 1-Epoch Training Trial & Checkpoint Verification

Runs a single complete epoch using the highest verified safe batch size from the benchmark. Confirms full metric generation and checkpoint reloadability.

In [ ]:
# Cell 7: Controlled 1-Epoch Trial
passing_bs = [r['batch_size'] for r in benchmark_results if r['status'] == 'PASS']
assert len(passing_bs) > 0, 'No batch sizes passed the memory benchmark!'
selected_bs = max(passing_bs)
print(f'Selected Batch Size for Controlled Trial: {selected_bs}')

trial_cmd = [
    sys.executable, 'scripts/train.py',
    '--model', 'cxformer_small',
    '--data_dir', DATA_ROOT,
    '--model_dir', MODEL_DIR,
    '--batch_size', str(selected_bs),
    '--epochs', '1',
]

print('Executing 1-Epoch Trial:', ' '.join(trial_cmd))
trial_res = subprocess.run(trial_cmd, capture_output=True, text=True)
print(trial_res.stdout)
if trial_res.stderr:
    print('Trial STDERR:\n', trial_res.stderr)

assert trial_res.returncode == 0, f'Trial failed with return code {trial_res.returncode}'
print('[PASS] 1-Epoch Controlled Training Trial succeeded!')


In [ ]:
# Cell 8: Checkpoint Reload & Inference State Verification
best_ckpt = os.path.join(WORK_DIR, 'checkpoints', 'cxformer_small_best.pth')
latest_ckpt = os.path.join(WORK_DIR, 'checkpoints', 'cxformer_small_latest.pth')
target_ckpt = best_ckpt if os.path.isfile(best_ckpt) else latest_ckpt

assert os.path.isfile(target_ckpt), f'No checkpoint found at {target_ckpt}!'
ckpt_data = torch.load(target_ckpt, map_location=device, weights_only=False)
print(f'[PASS] Successfully loaded checkpoint: {target_ckpt}')
print(f'Epoch              : {ckpt_data.get("epoch")}')
print(f'Best Metric Value  : {ckpt_data.get("best_metric_value")}')
print(f'Validation Metrics : {ckpt_data.get("val_metrics")}')

# Verify clean state_dict reload into fresh model
eval_model = build_cxformer_small(model_dir=MODEL_DIR, num_classes=2, freeze_backbone=True).to(device)
eval_model.load_state_dict(ckpt_data['model_state_dict'])
eval_model.eval()

# Verify inference with reloaded model
sample_batch = sample_tensor.unsqueeze(0).to(device)
with torch.no_grad():
    test_logits = eval_model(sample_batch)

print(f'Inference Logits on Sample Image: {test_logits.cpu().numpy()}')
assert test_logits.shape == (1, 2)
assert torch.isfinite(test_logits).all().item() is True
print('[PASS] Checkpoint reload and forward inference verified bitwise.')
